<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Demo - Build and Register a Retrieval Pipeline

## Overview
A Retrieval-Augmented Generation (RAG) pipeline grounds a large language model in your own data so it can answer questions with accurate, up-to-date, domain-specific context. This lesson first introduces the fundamentals of RAG and the role of vector databases, then has you build a working pipeline: you will create a Databricks AI Search index over a product dataset, retrieve similar documents, assemble a RAG model, and register it to the Unity Catalog Model Registry.

The RAG pipeline will function as a simple **product design chatbot**. As contextual information, we will provide product descriptions previously listed on the Etsy website. Please note that the dataset we will use is publicly available, and the large language model (LLM) might already include this data in its training set. Consequently, the quality of responses with and without contextual data might not differ significantly. In a real-world scenario, the contextual data would include information that is new to the LLM.

## Learning Objectives
By the end of this demo, you will be able to:
- Describe the core components of the RAG architecture.
- Identify use cases where vector databases are valuable.
- Describe the features and benefits of Databricks AI Search.
- Configure an AI Search index using an existing Unity Catalog managed table.
- Retrieve documents from AI Search using a similarity search.
- Assemble a RAG pipeline by integrating multiple components.
- Register a RAG pipeline in the Unity Catalog Model Registry.

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, ML Environment Version 5 (ML v5)</strong>: How to select an environment version (<a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">GCP</a>)</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless compute with the ML v5 environment</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

To select the **ML v5** environment:

1. Click the **Environment** button on the right sidebar
2. Open the **Environment version** dropdown
3. Select **ML v5**

If you do not see **Serverless** available, contact your workspace administrator.

## A. Retrieval-Augmented Generation (RAG) Fundamentals

Before building the pipeline, let's establish what RAG is and why it is useful. These concepts underpin everything you build in this lesson and are referenced again in the final lab.

### A1. What is RAG?

<div class="r1">
  <style>
    @import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700;800&display=swap');
    .r1{font-family:'DM Sans','Segoe UI',Arial,sans-serif;max-width:1040px;margin:14px auto 6px;color:#1B3139;}
    .r1 *{box-sizing:border-box;}
    .r1-row{display:flex;flex-wrap:wrap;gap:22px;align-items:center;}
    .r1-col{flex:1 1 320px;}
        .r1 .xl{list-style:none;margin:6px 0 0;padding:0;}
    .r1 .xl li{position:relative;padding-left:26px;font-size:14.5px;line-height:1.5;color:#3a4d54;margin-bottom:12px;}
    .r1 .xl li b{color:#1B3139;}
    .r1 .xl li::before{content:'';position:absolute;left:0;top:2px;width:17px;height:17px;border-radius:50%;background:#4299E0;}
    .r1 .xl li::after{content:'';position:absolute;left:5px;top:6px;width:4px;height:7px;border:solid #fff;border-width:0 2px 2px 0;transform:rotate(45deg);}
    .r1-diag{flex:1 1 320px;background:linear-gradient(180deg,#FBFAF8,#F4F7F9);border:1px solid #EEEDE9;border-radius:14px;box-shadow:0 6px 16px rgba(27,49,57,.08);padding:16px;}
    .r1-diag svg{width:100%;height:auto;max-width:380px;display:block;margin:0 auto;}
  </style>
  <div class="r1-row">
    <div class="r1-col">
      <p style="font-size:15px;color:#3a4d54;"><b>R</b>etrieval <b>A</b>ugmented <b>G</b>eneration (RAG):</p>
      <ul class="xl">
        <li>A pattern (<a href="https://arxiv.org/abs/2005.11401" target="_blank">Lewis et al. 2020</a>) that improves the <b>efficacy</b> of LLM applications by <b>leveraging custom data</b>.</li>
        <li>Works by <b>retrieving</b> documents relevant to a question and providing them as context to <b>augment</b> the prompt sent to an LLM, improving <b>generation</b>.</li>
      </ul>
    </div>
    <div class="r1-diag">
      <svg viewBox="0 0 360 300" role="img" aria-label="RAG flow: a query and retrieved domain-specific context are combined into a prompt sent to an LLM">
        <defs><marker id="r1a" markerWidth="11" markerHeight="11" refX="6" refY="5" orient="auto"><path d="M0 0l8 5-8 5z" fill="#FF5F46"/></marker>
        <filter id="r1s" x="-20%" y="-20%" width="140%" height="140%"><feDropShadow dx="0" dy="3" stdDeviation="5" flood-color="#1B3139" flood-opacity="0.14"/></filter></defs>
        <g filter="url(#r1s)"><rect x="196" y="16" width="120" height="46" rx="9" fill="#fff" stroke="#4299E0" stroke-width="2"/></g>
        <text x="256" y="44" text-anchor="middle" font-size="16" font-weight="800" fill="#2272B4" font-family="DM Sans,sans-serif">Query</text>
        <line x1="256" y1="62" x2="256" y2="104" stroke="#FF5F46" stroke-width="2.4" marker-end="url(#r1a)"/>
        <g filter="url(#r1s)"><rect x="8" y="120" width="150" height="70" rx="12" fill="#EAF2FB" stroke="#4299E0" stroke-width="1.6"/></g>
        <text x="83" y="150" text-anchor="middle" font-size="13.5" font-weight="700" fill="#1B3139" font-family="DM Sans,sans-serif">Retrieve domain-</text>
        <text x="83" y="168" text-anchor="middle" font-size="13.5" font-weight="700" fill="#1B3139" font-family="DM Sans,sans-serif">specific context</text>
        <line x1="158" y1="150" x2="192" y2="150" stroke="#FF5F46" stroke-width="2.4" marker-end="url(#r1a)"/>
        <g filter="url(#r1s)"><rect x="196" y="108" width="120" height="90" rx="9" fill="#fff" stroke="#1B5162" stroke-width="2"/></g>
        <text x="210" y="130" font-size="14" font-weight="800" fill="#1B3139" font-family="DM Sans,sans-serif">Prompt</text>
        <text x="210" y="150" font-size="12.5" fill="#3a4d54" font-family="DM Sans,sans-serif">Query</text>
        <text x="210" y="168" font-size="12.5" fill="#3a4d54" font-family="DM Sans,sans-serif">Similar doc 1</text>
        <text x="210" y="186" font-size="12.5" fill="#3a4d54" font-family="DM Sans,sans-serif">Similar doc 2 …</text>
        <line x1="256" y1="198" x2="256" y2="234" stroke="#FF5F46" stroke-width="2.4" marker-end="url(#r1a)"/>
        <g filter="url(#r1s)"><rect x="206" y="240" width="100" height="50" rx="14" fill="#1B5162"/></g>
        <text x="256" y="270" text-anchor="middle" font-size="17" font-weight="800" fill="#fff" font-family="DM Sans,sans-serif">LLM</text>
      </svg>
    </div>
  </div>
  <hr style="border:none;border-top:1px solid #EEEDE9;margin:20px 0;">
  <ul class="xl">
    <li>The main problem RAG solves is the <b>knowledge gap</b> &mdash; it enhances the accuracy and relevance of responses.</li>
    <li>Common use cases: <b>Q&amp;A chatbots</b> (accurate answers from internal sources), <b>search augmentation</b> (LLM-generated answers alongside search), and <b>content creation &amp; summarization</b> using additional context.</li>
  </ul>
</div>

### A2. Benefits of RAG Architecture

<div class="b4">
  <style>
    @import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700;800&display=swap');
    .b4{font-family:'DM Sans','Segoe UI',Arial,sans-serif;max-width:1040px;margin:14px auto 6px;color:#1B3139;}
    .b4 *{box-sizing:border-box;}
    .b4-grid{display:grid;grid-template-columns:repeat(2,1fr);gap:18px;}
    .b4-card{background:#fff;border-radius:15px;border-top:5px solid var(--acc);box-shadow:0 6px 16px rgba(27,49,57,.08);padding:20px;}
    .b4-badge{width:50px;height:50px;border-radius:13px;background:var(--tint);color:var(--acc);display:inline-flex;align-items:center;justify-content:center;margin-bottom:12px;}
    .b4-badge svg{width:27px;height:27px;}
    .b4-card h4{margin:0 0 8px;font-size:16px;font-weight:800;color:#0b2026;}
    .b4-card ul{list-style:none;margin:0;padding:0;}
    .b4-card li{position:relative;padding-left:18px;font-size:13.5px;line-height:1.5;color:#3a4d54;margin-bottom:8px;}
    .b4-card li b{color:#1B3139;}
    .b4-card li::before{content:'';position:absolute;left:2px;top:7px;width:7px;height:7px;border-radius:2px;transform:rotate(45deg);background:var(--acc);}
    @media(max-width:640px){.b4-grid{grid-template-columns:1fr;}}
  </style>
  <div class="b4-grid">
    <div class="b4-card" style="--acc:#00A972;--tint:rgba(0,169,114,.12);"><div class="b4-badge"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><path d="M20 11a8 8 0 10-1.5 5"/><path d="M20 5v6h-6"/></svg></div><h4>Up-to-date &amp; accurate responses</h4><ul><li>Responses are not based solely on static training data.</li><li>The model uses <b>up-to-date external data sources</b>.</li></ul></div>
    <div class="b4-card" style="--acc:#4299E0;--tint:rgba(66,153,224,.12);"><div class="b4-badge"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><path d="M12 3l7 2.8v5.2c0 4.4-3 7.4-7 8.9-4-1.5-7-4.5-7-8.9V5.8L12 3Z"/><path d="M8.8 11.8l2.2 2.2 4.2-4.4"/></svg></div><h4>Reduced hallucinations</h4><ul><li>Mitigates the risk of <b>hallucinations</b> or incorrect information.</li><li>Outputs can cite <b>original sources</b> for human verification.</li></ul></div>
    <div class="b4-card" style="--acc:#FFAB00;--tint:rgba(255,171,0,.14);"><div class="b4-badge"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><circle cx="12" cy="12" r="8.5"/><circle cx="12" cy="12" r="4.5"/><circle cx="12" cy="12" r="1"/></svg></div><h4>Domain-specific contextualization</h4><ul><li>Interfaces with <b>proprietary or domain-specific</b> data.</li><li>Improves accuracy using contextually relevant information.</li></ul></div>
    <div class="b4-card" style="--acc:#FF5F46;--tint:rgba(255,95,70,.12);"><div class="b4-badge"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><ellipse cx="9" cy="6" rx="6" ry="2.6"/><path d="M3 6v4c0 1.4 2.7 2.6 6 2.6s6-1.2 6-2.6V6"/><ellipse cx="15" cy="15" rx="6" ry="2.6"/><path d="M9 15v3c0 1.4 2.7 2.6 6 2.6s6-1.2 6-2.6v-3"/></svg></div><h4>Efficiency &amp; cost-effectiveness</h4><ul><li>In-context learning <b>without the up-front overhead of fine-tuning</b>.</li><li>Ideal where models need frequent updates with new data.</li></ul></div>
  </div>
</div>

### A3. A Sample RAG Architecture

<div class="ar">
  <style>
    @import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700;800&display=swap');
    .ar{font-family:'DM Sans','Segoe UI',Arial,sans-serif;max-width:1040px;margin:14px auto 6px;color:#1B3139;}
    .ar *{box-sizing:border-box;}
    .ar-zone{background:#fff;border:1px solid #EEEDE9;border-radius:14px;box-shadow:0 5px 14px rgba(27,49,57,.07);padding:16px 18px;margin-bottom:14px;}
    .ar-zt{display:inline-block;font-size:12px;font-weight:800;letter-spacing:.5px;text-transform:uppercase;color:#fff;background:var(--zc);border-radius:7px;padding:5px 12px;margin-bottom:14px;}
    .ar-flow{display:flex;flex-wrap:wrap;align-items:center;gap:10px;}
    .ar-step{flex:1 1 130px;display:flex;flex-direction:column;align-items:center;gap:8px;text-align:center;background:var(--tint);border-radius:11px;padding:14px 10px;font-size:12.5px;font-weight:700;color:#1B3139;}
    .ar-ic{width:40px;height:40px;border-radius:11px;background:#fff;color:var(--acc);display:inline-flex;align-items:center;justify-content:center;box-shadow:0 1px 4px rgba(27,49,57,.12);}
    .ar-ic svg{width:23px;height:23px;}
    .ar-arw{flex:0 0 auto;color:#9AA6A0;font-size:20px;font-weight:800;}
  </style>
  <div class="ar-zone" style="--zc:#4299E0;">
    <span class="ar-zt" style="--zc:#4299E0;">1 · Document Embedding</span>
    <div class="ar-flow">
      <div class="ar-step" style="--acc:#4299E0;--tint:rgba(66,153,224,.12);"><span class="ar-ic"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><path d="M6 3h8l4 4v14H6z"/><path d="M14 3v4h4"/><path d="M9 12h6M9 15h6"/></svg></span><span>Source documents</span></div><span class="ar-arw">&rarr;</span>
      <div class="ar-step" style="--acc:#4299E0;--tint:rgba(66,153,224,.12);"><span class="ar-ic"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><path d="M9 4.5a2.6 2.6 0 00-2.6 2.6 2.4 2.4 0 00-1.4 4.3A2.6 2.6 0 006 15.9a2.4 2.4 0 003 2.6 2.4 2.4 0 003-.6 2.4 2.4 0 003 .6 2.4 2.4 0 003-2.6 2.6 2.6 0 001-4.5 2.4 2.4 0 00-1.4-4.3A2.6 2.6 0 0015 4.5a2.5 2.5 0 00-3 .9 2.5 2.5 0 00-3-.9Z"/></svg></span><span>Embedding model</span></div><span class="ar-arw">&rarr;</span>
      <div class="ar-step" style="--acc:#4299E0;--tint:rgba(66,153,224,.12);"><span class="ar-ic"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><ellipse cx="12" cy="6" rx="7" ry="3"/><path d="M5 6v12c0 1.7 3.1 3 7 3s7-1.3 7-3V6"/><path d="M5 12c0 1.7 3.1 3 7 3s7-1.3 7-3"/></svg></span><span>Embeddings in vector store</span></div>
    </div>
  </div>
  <div class="ar-zone" style="--zc:#FFAB00;">
    <span class="ar-zt" style="--zc:#C7860A;">2 · User Query &amp; Retrieval</span>
    <div class="ar-flow">
      <div class="ar-step" style="--acc:#FFAB00;--tint:rgba(255,171,0,.14);"><span class="ar-ic"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><circle cx="12" cy="8" r="3.4"/><path d="M5.5 20a6.5 6.5 0 0113 0"/></svg></span><span>User asks a question</span></div><span class="ar-arw">&rarr;</span>
      <div class="ar-step" style="--acc:#FFAB00;--tint:rgba(255,171,0,.14);"><span class="ar-ic"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><path d="M9 4.5a2.6 2.6 0 00-2.6 2.6 2.4 2.4 0 00-1.4 4.3A2.6 2.6 0 006 15.9a2.4 2.4 0 003 2.6 2.4 2.4 0 003-.6 2.4 2.4 0 003 .6 2.4 2.4 0 003-2.6 2.6 2.6 0 001-4.5 2.4 2.4 0 00-1.4-4.3A2.6 2.6 0 0015 4.5a2.5 2.5 0 00-3 .9 2.5 2.5 0 00-3-.9Z"/></svg></span><span>Query &rarr; embedding</span></div><span class="ar-arw">&rarr;</span>
      <div class="ar-step" style="--acc:#FFAB00;--tint:rgba(255,171,0,.14);"><span class="ar-ic"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><circle cx="11" cy="11" r="6"/><path d="M20 20l-4.5-4.5"/></svg></span><span>Similarity search</span></div>
    </div>
  </div>
  <div class="ar-zone" style="--zc:#00A972;">
    <span class="ar-zt" style="--zc:#00A972;">3 · Generation</span>
    <div class="ar-flow">
      <div class="ar-step" style="--acc:#00A972;--tint:rgba(0,169,114,.12);"><span class="ar-ic"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><path d="M4 5h16v11H9l-5 4V5z"/><path d="M12 8v5M9.5 10.5h5"/></svg></span><span>Retrieved context as prompt</span></div><span class="ar-arw">&rarr;</span>
      <div class="ar-step" style="--acc:#00A972;--tint:rgba(0,169,114,.12);"><span class="ar-ic"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><path d="M9 4.5a2.6 2.6 0 00-2.6 2.6 2.4 2.4 0 00-1.4 4.3A2.6 2.6 0 006 15.9a2.4 2.4 0 003 2.6 2.4 2.4 0 003-.6 2.4 2.4 0 003 .6 2.4 2.4 0 003-2.6 2.6 2.6 0 001-4.5 2.4 2.4 0 00-1.4-4.3A2.6 2.6 0 0015 4.5a2.5 2.5 0 00-3 .9 2.5 2.5 0 00-3-.9Z"/></svg></span><span>Generation model</span></div><span class="ar-arw">&rarr;</span>
      <div class="ar-step" style="--acc:#00A972;--tint:rgba(0,169,114,.12);"><span class="ar-ic"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><path d="M20 5H4v11h5l5 4v-4h6z"/><path d="M8 10.5l2.5 2.5 4-4.5"/></svg></span><span>User receives an answer</span></div>
    </div>
  </div>
</div>

### A4. Main Concepts of a RAG Workflow

<div class="c6">
  <style>
    @import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700;800&display=swap');
    .c6{font-family:'DM Sans','Segoe UI',Arial,sans-serif;max-width:1040px;margin:14px auto 6px;color:#1B3139;}
    .c6 *{box-sizing:border-box;}
    .c6-grid{display:grid;grid-template-columns:repeat(3,1fr);gap:16px;}
    .c6-tile{background:#fff;border:1px solid #EEEDE9;border-top:5px solid var(--acc);border-radius:14px;box-shadow:0 5px 14px rgba(27,49,57,.07);padding:16px 16px 14px;}
    .c6-head{display:flex;align-items:center;gap:10px;margin-bottom:9px;}
    .c6-ic{width:38px;height:38px;border-radius:10px;background:var(--tint);color:var(--acc);display:inline-flex;align-items:center;justify-content:center;}
    .c6-ic svg{width:22px;height:22px;}
    .c6-head b{font-size:14.5px;font-weight:800;color:#0b2026;}
    .c6-tile p{margin:0;font-size:13px;line-height:1.5;color:#3a4d54;}
    @media(max-width:720px){.c6-grid{grid-template-columns:1fr;}}
  </style>
  <div class="c6-grid">
    <div class="c6-tile" style="--acc:#4299E0;--tint:rgba(66,153,224,.12);"><div class="c6-head"><span class="c6-ic"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><rect x="3" y="4" width="8" height="8" rx="1.5"/><circle cx="17" cy="7" r="2"/><circle cx="17" cy="16" r="2"/><circle cx="7.5" cy="18" r="2"/><path d="M11 8h4M11.5 10.5l3.8 4.2M9 12v4"/></svg></span><b>Index &amp; Embed</b></div><p>An embedding model creates <b>vector representations</b> of documents and user queries.</p></div>
    <div class="c6-tile" style="--acc:#00A972;--tint:rgba(0,169,114,.12);"><div class="c6-head"><span class="c6-ic"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><ellipse cx="12" cy="6" rx="7" ry="3"/><path d="M5 6v12c0 1.7 3.1 3 7 3s7-1.3 7-3V6"/><path d="M5 12c0 1.7 3.1 3 7 3s7-1.3 7-3"/></svg></span><b>Vector Store</b></div><p>Stores unstructured data indexed by vectors &mdash; a vector DB, library, or plugin.</p></div>
    <div class="c6-tile" style="--acc:#FFAB00;--tint:rgba(255,171,0,.14);"><div class="c6-head"><span class="c6-ic"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><circle cx="11" cy="11" r="6"/><path d="M20 20l-4.5-4.5"/></svg></span><b>Retrieval</b></div><p>Searches stored vectors with similarity search to <b>retrieve relevant information</b>.</p></div>
    <div class="c6-tile" style="--acc:#FF5F46;--tint:rgba(255,95,70,.12);"><div class="c6-head"><span class="c6-ic"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><path d="M4 7h10M4 12h7M4 17h13"/><path d="M17 5l3 3-3 3"/></svg></span><b>Filtering &amp; Reranking</b></div><p>Selects or ranks retrieved documents before passing them as context.</p></div>
    <div class="c6-tile" style="--acc:#1B5162;--tint:rgba(27,81,98,.11);"><div class="c6-head"><span class="c6-ic"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><path d="M4 5h16v11H9l-5 4V5z"/><path d="M12 8v5M9.5 10.5h5"/></svg></span><b>Prompt Augmentation</b></div><p>Injects retrieved data into the prompt to <b>enhance context</b>.</p></div>
    <div class="c6-tile" style="--acc:#98102A;--tint:rgba(152,16,42,.10);"><div class="c6-head"><span class="c6-ic"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><path d="M12 3v18M3 12h18M6 6l12 12M18 6L6 18"/></svg></span><b>Generation</b></div><p>An LLM generates the response to the user’s request.</p></div>
  </div>
</div>

## B. Classroom Setup

Let's set up your environment. This installs the libraries and datasets you need for the demo, then defines key variables for the AI Search endpoint, source table, and index table.

In [0]:
%run ../Includes/Classroom-Setup-02

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.



The examples and models presented in this course are intended solely for demonstration and educational purposes.
 Please note that the models and prompt examples may sometimes contain offensive, inaccurate, biased, or harmful content.


Validation of table learn_get_started_genai_puli_reddy.default.e_commerce_product_listings_dataset complete. No errors found.


DataFrame[]

This script defines key variables for the AI Search endpoint, source table, and index table. These variables will be used throughout the demo to reference resources in your workspace.

In [0]:
# assign vs search endpoint by username
vs_endpoint_name = DA.vector_search_endpoint_name
vs_source_table_name = f"{DA.catalog_name}.{DA.schema_name}.product_text"
vs_index_table_name = f"{DA.catalog_name}.{DA.schema_name}.product_embeddings"

print(f"=== Variables that you will need for this demo === \n")
print(f"Catalog Name                : {DA.catalog_name}\n")
print(f"Schema Name                 : {DA.schema_name}\n")
print(f"Assigned VS endpoint name   : {vs_endpoint_name} \n")
print(f"VS source table name        : {vs_source_table_name} \n")
print(f"VS index table name         : {vs_index_table_name } \n")

=== Variables that you will need for this demo === 

Catalog Name                : learn_get_started_genai_puli_reddy

Schema Name                 : default

Assigned VS endpoint name   : learn_academy_search 

VS source table name        : learn_get_started_genai_puli_reddy.default.product_text 

VS index table name         : learn_get_started_genai_puli_reddy.default.product_embeddings 



## C. Load the Dataset

We will use a real-world product dataset. For this demo, we focus on the product title and description, which are most useful for finding similar products. Run the next cell to preview the source table.

In [0]:
display(spark.sql(f"SELECT * FROM {vs_source_table_name}"))

id,document
H4HMD7T009,"## Title: Ergonomic Mouse ## Description: The Ergonomic Mouse offered by PrimeLabs is shaped around the idea that a useful product does not need to be complicated to be effective. It is meant for people who want something they can set up once and then trust during regular work or home activities. The form factor is kept modest so that it can fit on a desk, shelf, or counter without demanding extra space or attention. During typical use, it aims to provide consistent behavior, so you can focus on your tasks and not on the tool itself. Rather than specializing in a narrow scenario, it is built to handle a range of everyday situations at a steady level of quality. Features such as Sturdy construction; Balanced design; Straightforward handling are included to support this general-purpose approach, offering a mix of practicality, familiarity, and low-effort upkeep. The result is a product that is intended to quietly support your routine instead of becoming another source of complexity. ## URL: https://dealstreak.io/product/sleek-gear-9856/"
96NCWMLFCI,"## Title: UltraSlim Keyboard ## Description: The UltraSlim Keyboard offered by PolarSelect is shaped around the idea that a useful product does not need to be complicated to be effective. It is meant for people who want something they can set up once and then trust during regular work or home activities. The form factor is kept modest so that it can fit on a desk, shelf, or counter without demanding extra space or attention. During typical use, it aims to provide consistent behavior, so you can focus on your tasks and not on the tool itself. Rather than specializing in a narrow scenario, it is built to handle a range of everyday situations at a steady level of quality. Features such as Sturdy construction; Balanced design; Straightforward handling are included to support this general-purpose approach, offering a mix of practicality, familiarity, and low-effort upkeep. The result is a product that is intended to quietly support your routine instead of becoming another source of complexity. ## URL: https://urbangoods.shop/product/ultra-item-6281/"
AU63RT0O93,"## Title: Rechargeable Juicer ## Description: NovaLabs's Rechargeable Juicer is designed around everyday patterns of use, where reliability and clarity matter more than constant adjustment. The intent is to provide something that feels straightforward from the moment you take it out of the box, with controls and behavior that are easy to understand. Its visual style is kept neutral so that it can blend into different environments without looking out of place. Over time, it is meant to remain stable, so that you do not have to continuously relearn how to interact with it. Instead of chasing short-lived trends, the design focuses on steady function and predictable responses. The inclusion of Non-slip base; Practical layout; Optimized for regular workloads is aimed at supporting daily tasks while keeping the overall experience calm and manageable. In this way, the Rechargeable Juicer is positioned as a supportive background tool, helping you maintain momentum in your routine without demanding extra attention. ## URL: https://trendifyonline.net/product/premium-device-3302/"
DGJE85BFWT,"## Title: Portable Microphone ## Description: The Portable Microphone offered by LumaHub is shaped around the idea that a useful product does not need to be complicated to be effective. It is meant for people who want something they can set up once and then trust during regular work or home activities. The form factor is kept modest so that it can fit on a desk, shelf, or counter without demanding extra space or attention. During typical use, it aims to provide consistent behavior, so you can focus on your tasks and not on the tool itself. Rather than specializing in a narrow scenario, it is built to handle a range of everyday situations at a steady level of quality. Features such as Compact form factor; Simple controls; Works we

## D. Databricks AI Search

To retrieve relevant context for the RAG pipeline, we use **Databricks AI Search** — Databricks' integrated similarity-search solution. The following sections explain vector databases and AI Search before you create an index over the dataset you just loaded.


### D1. What are Vector Databases?


- A vector database is a database that is optimized to **store and retrieve high-dimensional vectors such as embeddings**.

- In RAG architecture **contextual information** is stored in vectors.

- Vector databases are designed for **efficient storage of vectors** utilized in generative AI applications, which rely on identifying documents or images with similarities.

- Vector databases provide a query interface that retrieves vectors most similar to a specified query vector.

### D2. Why are Vector Databases So Hot?

<div class="d2">
  <style>
    @import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700;800&display=swap');
    .d2{font-family:'DM Sans','Segoe UI',Arial,sans-serif;max-width:1040px;margin:14px auto 6px;color:#1B3139;}
    .d2 *{box-sizing:border-box;}
    .d2-row{display:flex;flex-wrap:wrap;gap:22px;align-items:center;}
    .d2-col{flex:1 1 320px;}
        .d2 .xl{list-style:none;margin:6px 0 0;padding:0;}
    .d2 .xl li{position:relative;padding-left:26px;font-size:14.5px;line-height:1.5;color:#3a4d54;margin-bottom:12px;}
    .d2 .xl li b{color:#1B3139;}
    .d2 .xl li::before{content:'';position:absolute;left:0;top:2px;width:17px;height:17px;border-radius:50%;background:#4299E0;}
    .d2 .xl li::after{content:'';position:absolute;left:5px;top:6px;width:4px;height:7px;border:solid #fff;border-width:0 2px 2px 0;transform:rotate(45deg);}
    .d2-frame{flex:1 1 320px;background:#fff;border:1px solid #EEEDE9;border-radius:14px;padding:12px;box-shadow:0 6px 16px rgba(27,49,57,.08);text-align:center;}
    .d2-frame img{display:block;width:100%;height:auto;border-radius:8px;}
    .d2-frame .src{font-size:11.5px;color:#5A6F77;margin-top:8px;}
    .d2-frame .src a{color:#2272B4;}
  </style>
  <div class="d2-row">
    <div class="d2-col">
      <ul class="xl">
        <li>Specialized, full-fledged databases for <b>unstructured data</b> &mdash; they inherit database properties (CRUD).</li>
        <li>Speed up the search for the closest vectors using AI search algorithms such as <b>Approximate Nearest Neighbor (ANN)</b>.</li>
        <li>Organize embeddings into <b>indices</b> for scalable, low-latency retrieval.</li>
      </ul>
    </div>
    <div class="d2-frame">
      <img src="../Includes/images/M3L2_44.png" alt="Illustration of vector embeddings organized for similarity search">
      <div class="src">Image source: <a href="https://weaviate.io/blog/vector-embeddings-explained" target="_blank">weaviate</a></div>
    </div>
  </div>
</div>

### D3. Databricks AI Search

<div class="ds">
  <style>
    @import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700;800&display=swap');
    .ds{font-family:'DM Sans','Segoe UI',Arial,sans-serif;max-width:1040px;margin:14px auto 6px;color:#1B3139;}
    .ds *{box-sizing:border-box;}
    .ds-sub{font-size:14px;color:#5A6F77;margin:0 0 16px;}
    .ds-row{display:flex;flex-wrap:wrap;gap:22px;align-items:center;}
    .ds-col{flex:1 1 320px;}
        .ds .xl{list-style:none;margin:6px 0 0;padding:0;}
    .ds .xl li{position:relative;padding-left:26px;font-size:14.5px;line-height:1.5;color:#3a4d54;margin-bottom:12px;}
    .ds .xl li b{color:#1B3139;}
    .ds .xl li::before{content:'';position:absolute;left:0;top:2px;width:17px;height:17px;border-radius:50%;background:#4299E0;}
    .ds .xl li::after{content:'';position:absolute;left:5px;top:6px;width:4px;height:7px;border:solid #fff;border-width:0 2px 2px 0;transform:rotate(45deg);}
    .ds-diag{flex:1 1 300px;background:linear-gradient(180deg,#FBFAF8,#F4F7F9);border:1px solid #EEEDE9;border-radius:14px;box-shadow:0 6px 16px rgba(27,49,57,.08);padding:16px;}
    .ds-diag svg{width:100%;height:auto;max-width:330px;display:block;margin:0 auto;}
  </style>
  <p class="ds-sub">A vector database built into the <b>Databricks Data + AI Platform</b>.</p>
  <div class="ds-row">
    <div class="ds-col">
      <ul class="xl">
        <li>Stores vector representations of your data, plus metadata.</li>
        <li>Tightly integrated with your Lakehouse.</li>
        <li>Scalable, low-latency production service with <b>zero operational overhead</b>.</li>
        <li>Supports ACLs via <b>Unity Catalog</b> integration.</li>
        <li>API for real-time similarity search (REST API &amp; Python client), with metadata filters.</li>
      </ul>
    </div>
    <div class="ds-diag">
      <svg viewBox="0 0 320 320" role="img" aria-label="Databricks AI Search engine with an indexer and vector database, queried via REST API or Python SDK and auto-synced from a source Unity Catalog managed table">
        <defs><marker id="dsa" markerWidth="11" markerHeight="11" refX="6" refY="5" orient="auto"><path d="M0 0l8 5-8 5z" fill="#00A972"/></marker>
        <marker id="dsb" markerWidth="11" markerHeight="11" refX="6" refY="5" orient="auto"><path d="M0 0l8 5-8 5z" fill="#4299E0"/></marker>
        <filter id="dss" x="-20%" y="-20%" width="140%" height="140%"><feDropShadow dx="0" dy="3" stdDeviation="5" flood-color="#1B3139" flood-opacity="0.14"/></filter></defs>
        <circle cx="160" cy="40" r="26" fill="#fff" stroke="#1B5162" stroke-width="2"/>
        <g transform="translate(148,28)" fill="none" stroke="#618794" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"><circle cx="12" cy="9" r="4"/><path d="M4 22a8 8 0 0116 0"/></g>
        <line x1="150" y1="68" x2="150" y2="104" stroke="#00A972" stroke-width="2.2" marker-end="url(#dsa)"/>
        <line x1="170" y1="104" x2="170" y2="68" stroke="#00A972" stroke-width="2.2" stroke-dasharray="5 4" marker-end="url(#dsa)"/>
        <text x="196" y="86" font-size="11.5" font-weight="700" fill="#00A972" font-family="DM Sans,sans-serif">REST API /</text>
        <text x="196" y="100" font-size="11.5" font-weight="700" fill="#00A972" font-family="DM Sans,sans-serif">Python SDK</text>
        <g filter="url(#dss)"><rect x="26" y="112" width="268" height="112" rx="14" fill="#FBF3D6" stroke="#D9C88E" stroke-width="1.4" stroke-dasharray="6 4"/></g>
        <text x="160" y="138" text-anchor="middle" font-size="15" font-weight="800" fill="#1B3139" font-family="DM Sans,sans-serif">Databricks AI Search</text>
        <g filter="url(#dss)"><rect x="44" y="152" width="106" height="56" rx="10" fill="#F6C99A" stroke="#D89B5C" stroke-width="1.2"/></g>
        <text x="97" y="185" text-anchor="middle" font-size="13.5" font-weight="800" fill="#7A4B16" font-family="DM Sans,sans-serif">Indexer</text>
        <g filter="url(#dss)"><ellipse cx="223" cy="164" rx="43" ry="10" fill="#C9BEEA" stroke="#9C8BD1" stroke-width="1.2"/><path d="M180 164v34c0 5.5 19.3 10 43 10s43-4.5 43-10v-34" fill="#C9BEEA" stroke="#9C8BD1" stroke-width="1.2"/></g>
        <text x="223" y="192" text-anchor="middle" font-size="12.5" font-weight="800" fill="#4B3B77" font-family="DM Sans,sans-serif">Vector DB</text>
        <line x1="160" y1="290" x2="160" y2="228" stroke="#4299E0" stroke-width="2.2" marker-end="url(#dsb)"/>
        <text x="196" y="262" font-size="12" font-weight="700" fill="#2272B4" font-family="DM Sans,sans-serif">Auto Sync</text>
        <text x="160" y="308" text-anchor="middle" font-size="13.5" font-weight="800" fill="#1B3139" font-family="DM Sans,sans-serif">Source Unity Catalog managed table</text>
      </svg>
    </div>
  </div>
</div>

### D4. How does AI Search Work?

<div class="m3">
  <style>
    @import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700;800&display=swap');
    .m3{font-family:'DM Sans','Segoe UI',Arial,sans-serif;max-width:1040px;margin:14px auto 6px;color:#1B3139;}
    .m3 *{box-sizing:border-box;}
    .m3-grid{display:grid;grid-template-columns:repeat(3,1fr);gap:16px;}
    .m3-card{background:#fff;border:1px solid #EEEDE9;border-top:5px solid var(--acc);border-radius:14px;box-shadow:0 5px 14px rgba(27,49,57,.07);padding:18px;}
    .m3-num{width:32px;height:32px;border-radius:50%;background:var(--acc);color:#fff;font-weight:800;font-size:15px;display:inline-flex;align-items:center;justify-content:center;margin-bottom:10px;}
    .m3-card h4{margin:0 0 10px;font-size:15px;font-weight:800;color:#0b2026;}
    .m3-card ul{list-style:none;margin:0;padding:0;}
    .m3-card li{position:relative;padding-left:18px;font-size:13px;line-height:1.5;color:#3a4d54;margin-bottom:9px;}
    .m3-card li b{color:#1B3139;}
    .m3-card li::before{content:'';position:absolute;left:2px;top:7px;width:7px;height:7px;border-radius:2px;transform:rotate(45deg);background:var(--acc);}
    @media(max-width:720px){.m3-grid{grid-template-columns:1fr;}}
  </style>
  <div class="m3-grid">
    <div class="m3-card" style="--acc:#4299E0;--tint:rgba(66,153,224,.12);"><div class="m3-num">1</div><h4>Managed Embeddings</h4><ul><li><b>Databricks computes</b> the embeddings for your data using a model you specify.</li><li>The index <b>automatically syncs</b> with the Unity Catalog managed table as source data updates.</li></ul></div>
    <div class="m3-card" style="--acc:#00A972;--tint:rgba(0,169,114,.12);"><div class="m3-num">2</div><h4>Self-Managed Embeddings</h4><ul><li>You <b>provide pre-calculated embeddings</b> in your Unity Catalog managed table.</li><li>The index auto-syncs as the table updates &mdash; <b>you generate the embeddings</b>.</li></ul></div>
    <div class="m3-card" style="--acc:#FFAB00;--tint:rgba(255,171,0,.14);"><div class="m3-num">3</div><h4>Direct Access</h4><ul><li>You <b>manually update the index</b> via REST API or Python SDK when data changes.</li><li>Requires <b>manual intervention</b> to maintain synchronization.</li></ul></div>
  </div>
</div>

### D5. Set up AI Search

<div class="s3">
  <style>
    @import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700;800&display=swap');
    .s3{font-family:'DM Sans','Segoe UI',Arial,sans-serif;max-width:1040px;margin:14px auto 6px;color:#1B3139;}
    .s3 *{box-sizing:border-box;}
    .s3-grid{display:grid;grid-template-columns:repeat(3,1fr);gap:16px;}
    .s3-card{background:#fff;border:1px solid #EEEDE9;border-top:5px solid var(--acc);border-radius:14px;box-shadow:0 5px 14px rgba(27,49,57,.07);padding:18px;}
    .s3-head{display:flex;align-items:center;gap:10px;margin-bottom:10px;}
    .s3-num{width:30px;height:30px;border-radius:50%;background:var(--acc);color:#fff;font-weight:800;font-size:14px;display:inline-flex;align-items:center;justify-content:center;}
    .s3-ic{width:38px;height:38px;border-radius:10px;background:var(--tint);color:var(--acc);display:inline-flex;align-items:center;justify-content:center;}
    .s3-ic svg{width:22px;height:22px;}
    .s3-card h4{margin:0 0 10px;font-size:14.5px;font-weight:800;color:#0b2026;}
    .s3-card ul{list-style:none;margin:0;padding:0;}
    .s3-card li{position:relative;padding-left:18px;font-size:13px;line-height:1.5;color:#3a4d54;margin-bottom:9px;}
    .s3-card li b{color:#1B3139;}
    .s3-card li::before{content:'';position:absolute;left:2px;top:7px;width:7px;height:7px;border-radius:2px;transform:rotate(45deg);background:var(--acc);}
    @media(max-width:720px){.s3-grid{grid-template-columns:1fr;}}
  </style>
  <div class="s3-grid">
    <div class="s3-card" style="--acc:#00A972;--tint:rgba(0,169,114,.12);"><div class="s3-head"><span class="s3-num">1</span><span class="s3-ic"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><circle cx="12" cy="12" r="8.5"/><path d="M12 3.5v17M3.5 12h17"/><path d="M5.5 7.5c2 1.2 11 1.2 13 0M5.5 16.5c2-1.2 11-1.2 13 0"/></svg></span></div><h4>AI Search Endpoint</h4><ul><li>The <b>compute resource</b> associated with AI Search.</li><li>Scales automatically to the index size and concurrent requests.</li><li>Supports multiple compute types.</li></ul></div>
    <div class="s3-card" style="--acc:#4299E0;--tint:rgba(66,153,224,.12);"><div class="s3-head"><span class="s3-num">2</span><span class="s3-ic"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><path d="M7 18a4 4 0 01-.5-8 6 6 0 0111.5 1.5A3.5 3.5 0 0117 18H7Z"/><path d="M12 12v5M9.5 14.5 12 12l2.5 2.5"/></svg></span></div><h4>Model Serving Endpoint</h4><ul><li>Create if you want Databricks to compute the embeddings.</li><li>Supports Foundation Model API embeddings (e.g. BGE), <b>external models</b> (e.g. OpenAI ada-002), and <b>custom models</b>.</li></ul></div>
    <div class="s3-card" style="--acc:#FFAB00;--tint:rgba(255,171,0,.14);"><div class="s3-head"><span class="s3-num">3</span><span class="s3-ic"><svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.85" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><rect x="4" y="3.5" width="16" height="17" rx="2"/><path d="M8 8h8M8 12h8M8 16h5"/><circle cx="6" cy="8" r="0.5" fill="currentColor" stroke="none"/></svg></span></div><h4>AI Search Index</h4><ul><li>Created and <b>auto-synced</b> from a Unity Catalog managed table.</li><li>Optimized for real-time approximate nearest-neighbor search.</li><li><b>Governed by Unity Catalog</b> with index-level ACLs.</li></ul></div>
  </div>
</div>

### D6. Set up AI Search &mdash; Using the UI

<div class="d6">
  <style>
    @import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700;800&display=swap');
    .d6{font-family:'DM Sans','Segoe UI',Arial,sans-serif;max-width:1040px;margin:14px auto 6px;color:#1B3139;}
    .d6 .sub{font-size:14px;color:#5A6F77;margin:0 0 14px;}
    .d6-frame{background:#fff;border:1px solid #EEEDE9;border-radius:14px;padding:10px;box-shadow:0 8px 20px rgba(27,49,57,.12);}
    .d6-frame img{display:block;width:100%;height:auto;border-radius:8px;}
  </style>
  <p class="sub">You can create the index in the UI (shown here) or via the API.</p>
  <div class="d6-frame"><img src="../Includes/images/M3L2_48.png" alt="Screenshot of creating an AI Search index in the Databricks UI"></div>
</div>

## E. Create and Test the AI Search Index

Now that you understand how AI Search works, let's create an index over the `product_text` table and test it with a sample query.


### E1. Create an AI Search Index via the UI

Follow these steps to create the index:

1. Navigate to **Catalog** in the left sidebar and open your course catalog.
2. Open your schema. The schema name is printed near the top of this notebook.
3. Select the **`product_text`** table created earlier in the notebook.
4. In the upper-right corner, click **Create** and then select **AI Search index**.
5. In the create dialog, configure the index with these settings:
    - **Name**: use **`product_embeddings`**.
    - **Primary key**: select **`id`**.
    - **Embedding source**: select **Compute embeddings**.
        - **Embedding source column**: select **`document`**.
    - **AI Search endpoint**: select the endpoint assigned to you. Refer to the values printed near the top of this notebook.
    - **Sync mode**: select **Triggered**.
6. Under **Advanced settings**:
    - **Embedding model**: select **`databricks-gte-large-en`**.
    - Set **Columns to index** to include **`document`**.
7. Click **Create** to create the index.

You should see the new **`product_embeddings`** index appear in your schema once creation starts.


### E2. (Alternative) Create an AI Search Index via the API

For simplicity, we created the AI search index using the UI. However, this process can also be accomplished programmatically using the `databricks-sdk`. For more detailed instructions, please refer to the [create and query an index documentation](https://docs.databricks.com/en/generative-ai/create-query-vector-search.html#create-index-using-the-python-sdk).

### E3. Test the Index: Search for Products Similar to the Query

Before building the RAG pipeline, let's check if the index is ready. We will use the `similarity_search` function to search for products that are similar to the query text. A correct result returns four product documents related to the query.

In [0]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient(disable_notice=True)

question = "handmade lamp with remote control"

try:
    # get index created in the previous step
    index = vsc.get_index(vs_endpoint_name, vs_index_table_name)

    # search for similar documents
    results = index.similarity_search(
        query_text = question,
        columns=["document"],
        num_results=4
    )

    # show the results
    docs = results.get("result", {}).get("data_array", [])
    pprint(docs)

except Exception as e:
    print(f"Error occurred while loading the index. Did you create an index in the previous step?: {e}")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[['## Title: Adjustable Smart Lamp\n'
  '\n'
  '## Description: With the Adjustable Smart Lamp, ZenithLabs aims to offer a practical companion '
  'for routine tasks, whether you are at home or at work. The product is structured so that you '
  'can start using it with minimal preparation, and its proportions are chosen to fit comfortably '
  'into compact spaces. The interaction model is intentionally straightforward, reducing the need '
  'to consult detailed instructions once you have become familiar with it. During extended use, '
  'the goal is to keep things steady rather than surprising, allowing you to build habits around '
  'how it behaves. This makes it suitable for people who value calm, predictable tools over highly '
  'specialized equipment. The set of characterist


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #4299E0); color: white; padding: 12px 18px; cursor: pointer; font-weight: 600; font-size: 12pt; border-radius: 8px; user-select: none;">
If you receive multiple similar documents, how would you determine which ones to utilize as context for your model?
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 16px 20px; background: #F9F7F4; font-size: 12pt; line-height: 1.7; color: #333;">

<p style="margin-top: 10px;">Would you select the top two documents, or would you employ a different approach — such as filtering or reranking, introduced in section A4 — to identify the most significant ones? There is no single right answer; the best choice depends on your use case and the quality of the retrieved results.</p>

</div>
</details>

## F. Enable MLflow Tracing

Before we build our RAG pipeline, let's turn on MLflow tracing. This helps us automatically track and log what happens in our pipeline, making it easier to review and improve our work later. 

**Note:** MLflow logging is enabled by default, but we'll show how to enable it here for learning purposes.

In [0]:
import mlflow
mlflow.langchain.autolog()

2026/11/16 17:50:09 WARNING mlflow.utils.autologging_utils: MLflow langchain autologging is known to be compatible with 1.0.4 <= langchain, but the installed version is 0.3.27. If you encounter errors during autologging, try upgrading / downgrading langchain to a compatible version, or try upgrading MLflow.


## G. Build a RAG Model

Now it's time to put everything together! We'll build a simple RAG (Retrieval-Augmented Generation) pipeline that uses the documents we found with AI Search to help answer questions. This makes our model smarter by giving it extra context.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">You must pass the correct VS index name as an environment variable. It is used in the <code>agent.py</code> file to access the index.</p>
</div>
</div>
</div>

In [0]:
import os
os.environ["VS_INDEX_NAME"] = vs_index_table_name

With the index name set, load the agent from the `agent.py` file and test it with a sample request. Loading the agent code from a file keeps the pipeline modular and reusable for deployment. You should see a generated product title and description in the response.

In [0]:
# Load the agent code from the agent.py file. This approach allows you to modularize the agent logic and reuse it for deployment or further development.
from agent import AGENT

user_input = "Create a concise title and 250-word description for a handmade lamp with remote dimming."
request = {
    "input": [
        {"role": "user", "content": user_input}
    ]
}
resp = AGENT.predict(request)
print(resp)

/opt/databricks-environments/databricks-ml/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten


tool_choice=None truncation=None id=None created_at=None error=None incomplete_details=None instructions=None metadata=None model=None object='response' output=[OutputItem(type='message', id='78131406-c864-48e0-9e69-fd35a6917bc5', content=[{'text': "Title: Handcrafted Table Lamp with Remote Dimming - Warm Ambiance for Any Room\n\nDescription: Elevate your home decor with our artisanal table lamp, expertly crafted to provide a warm and inviting ambiance. Made from high-quality materials, this lamp features a sturdy base and a soft linen shade that diffuses a gentle, soothing light. The pièce de résistance is its remote dimming capability, allowing you to adjust the brightness to suit your mood or task at hand. Whether you're reading, relaxing, or entertaining, this lamp's versatility ensures the perfect lighting atmosphere.\n\nThe lamp's unique, handcrafted design makes it a one-of-a-kind piece that adds a touch of personality to any room. Its compact size makes it perfect for bedside t

Trace(trace_id='tr-a605b29a62c0242cc57ce97e7a3a1970')

## H. Register the RAG Model to Unity Catalog

Once your RAG pipeline is working, you can register it in Unity Catalog. **This makes it easy to manage, share, and deploy your model later.** Registering your model is a best practice for keeping your work organized and ready for production.

In [0]:
import mlflow
from mlflow.models.resources import (
    DatabricksVectorSearchIndex
)
from mlflow.models import infer_signature
from pkg_resources import get_distribution

# Set Model Registry URI to Unity Catalog
mlflow.set_registry_uri("databricks-uc")

model_name = f"{DA.catalog_name}.{DA.schema_name}.getstarted_genai_retrevial_demo"

# Set the agent
mlflow.models.set_model(AGENT)

# Register the assembled RAG model in Model Registry with Unity Catalog
with mlflow.start_run(run_name="genai_gs_demo_02_01") as run:
    model_info = mlflow.pyfunc.log_model(
        name="agent",
        python_model="agent.py",
        pip_requirements=[
            f"langchain=={get_distribution('langchain').version}",
            f"databricks-vectorsearch=={get_distribution('databricks-vectorsearch').version}",
            f"databricks_langchain=={get_distribution('databricks_langchain').version}",
            f"mlflow=={get_distribution('mlflow').version}"
        ],
        resources=[
            DatabricksVectorSearchIndex(index_name=vs_index_table_name)
        ]
    )
model_uri = f"runs:/{run.info.run_id}/{model_info.name}"
model_version = mlflow.register_model(model_uri, model_name)
print(f"Model registered with name: {model_name} and version: {model_version.version}")

/home/spark-07d00d74-185d-40db-8ce2-09/.ipykernel/67/command-5842534141863380-368267147:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution
🔗 View Logged Model at: https://dbc-af847cd2-6ea8.cloud.databricks.com/ml/experiments/4148048598456801/models/m-3105be65461a4f77a5a3a46223af2b6b?o=7474653968665155
2026/11/16 17:52:38 INFO mlflow.pyfunc: Predicting on input example to validate output
Successfully registered model 'learn_get_started_genai_puli_reddy.default.getstarted_genai_retrevial_demo'.
2026/11/16 17:52:55 WARNING mlflow.tracking._model_registry.fluent: Run with id 4e960d39e8bb42c6b3a4592b4d9c2043 has no artifacts at artifact path 'agent', registering model based on models:/m-3105be65461a4f77a5a3a46223af2b6b instead


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

🔗 Created version '1' of model 'learn_get_started_genai_puli_reddy.default.getstarted_genai_retrevial_demo': https://dbc-af847cd2-6ea8.cloud.databricks.com/explore/data/models/learn_get_started_genai_puli_reddy/default/getstarted_genai_retrevial_demo/version/1?o=7474653968665155


Model registered with name: learn_get_started_genai_puli_reddy.default.getstarted_genai_retrevial_demo and version: 1



<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Warning</strong>
<p style="margin: 8px 0 0 0; color: #333;"><strong>Do not delete</strong> the catalog or tables created in this demo. You will need them for upcoming exercises.</p>
</div>
</div>
</div>

## Conclusion

In this demo, we:

1. Created a managed AI search index using a Unity Catalog managed table and verified retrieval of relevant product descriptions.
2. Built a RAG pipeline that integrates AI Search for context retrieval and a generative model for response generation.
3. Used an agent-as-code approach by defining the agent logic in the `agent.py` file, enabling modular development and easy deployment.
4. Registered the RAG pipeline in the Unity Catalog Model Registry, making it available for versioning, management, and production deployment.

This modular, agent-as-code workflow allows you to maintain, test, and deploy your RAG solutions efficiently using Databricks best practices.

Key takeaways:
- AI Search turns a Unity Catalog managed table into a governed similarity-search index for retrieval.
- A RAG pipeline combines retrieved context with a generative model for grounded responses.
- The agent-as-code pattern (`agent.py`) keeps the pipeline modular and deployable.
- Registering the pipeline in Unity Catalog makes it versioned, governed, and production-ready.

In the next module, you will learn how to evaluate and productionize your registered RAG model — measuring its quality, then deploying it to a real-time serving endpoint.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>


